# XGBoost on PySpark - 二分类（信用卡欺诈检测）
分布式数据处理 → 特征工程 → 分布式训练 → 评估 → 预测

In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("XGBoost-Binary-Classification") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.cores", "1") \
    .config("spark.driver.memory", "2g") \
    .config("spark.task.cpus", "1") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 13:34:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_samples = 100000
n_features = 28

# 生成合成特征
X = np.random.randn(n_samples, n_features)
logits = -3 + 2 * X[:, 0] + 1.5 * X[:, 1] - X[:, 2] + 0.5 * X[:, 3]
probs = 1 / (1 + np.exp(-logits))
y = (probs > 0.5).astype(int)

# 添加少量噪声
flip_idx = np.random.choice(n_samples, size=int(n_samples * 0.01), replace=False)
y[flip_idx] = 1 - y[flip_idx]

print(f"Total samples: {n_samples}")
print(f"Positive (fraud): {y.sum()} ({y.mean()*100:.2f}%)")
print(f"Negative (normal): {(1-y).sum()} ({(1-y.mean())*100:.2f}%)")

# 构建DataFrame
columns = ["Time"] + [f"V{i}" for i in range(1, n_features+1)] + ["Amount", "Class"]
data = np.column_stack([
    np.arange(n_samples),
    X,
    np.abs(np.random.randn(n_samples) * 100 + 50),
    y
])

df_pd = pd.DataFrame(data, columns=columns)
df_pd["Class"] = df_pd["Class"].astype(int)

# 保存CSV
data_dir = "/home/feynman/work/data"
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, "creditcard.csv")
df_pd.to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")
print(f"File size: {os.path.getsize(csv_path) / 1024 / 1024:.2f} MB")
df_pd.head()

Total samples: 100000
Positive (fraud): 14464 (14.46%)
Negative (normal): 85536 (85.54%)
Saved to /home/feynman/work/data/creditcard.csv
File size: 55.12 MB


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,0.767435,-0.469474,...,1.465649,-0.225776,0.067528,-1.424748,-0.544383,0.110923,-1.150994,0.375698,79.534575,0
1,1.0,-0.600639,-0.291694,-0.601707,1.852278,-0.013497,-1.057711,0.822545,-1.220844,0.208864,...,0.343618,-1.763040,0.324084,-0.385082,-0.676922,0.611676,1.031000,0.931280,78.643578,0
2,2.0,-0.839218,-0.309212,0.331263,0.975545,-0.479174,-0.185659,-1.106335,-1.196207,0.812526,...,0.087047,-0.299007,0.091761,-1.987569,-0.219672,0.357113,1.477894,-0.518270,31.221224,0
3,3.0,-0.808494,-0.501757,0.915402,0.328751,-0.529760,0.513267,0.097078,0.968645,-0.702053,...,-0.161286,0.404051,1.886186,0.174578,0.257550,-0.074446,-1.918771,-0.026514,91.888621,0
4,4.0,0.060230,2.463242,-0.192361,0.301547,-0.034712,-1.168678,1.142823,0.751933,0.791032,...,-1.062304,0.473592,-0.919424,1.549934,-0.783253,-0.322062,0.813517,-1.230864,63.332089,1


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,0.767435,-0.469474,...,1.465649,-0.225776,0.067528,-1.424748,-0.544383,0.110923,-1.150994,0.375698,79.534575,0
1,1.0,-0.600639,-0.291694,-0.601707,1.852278,-0.013497,-1.057711,0.822545,-1.220844,0.208864,...,0.343618,-1.763040,0.324084,-0.385082,-0.676922,0.611676,1.031000,0.931280,78.643578,0
2,2.0,-0.839218,-0.309212,0.331263,0.975545,-0.479174,-0.185659,-1.106335,-1.196207,0.812526,...,0.087047,-0.299007,0.091761,-1.987569,-0.219672,0.357113,1.477894,-0.518270,31.221224,0
3,3.0,-0.808494,-0.501757,0.915402,0.328751,-0.529760,0.513267,0.097078,0.968645,-0.702053,...,-0.161286,0.404051,1.886186,0.174578,0.257550,-0.074446,-1.918771,-0.026514,91.888621,0
4,4.0,0.060230,2.463242,-0.192361,0.301547,-0.034712,-1.168678,1.142823,0.751933,0.791032,...,-1.062304,0.473592,-0.919424,1.549934,-0.783253,-0.322062,0.813517,-1.230864,63.332089,1


## 1. 分布式数据加载与探索

In [3]:
# 用Spark分布式读取CSV
df = spark.read.csv(
    f"file://{csv_path}",
    header=True,
    inferSchema=True
)

print(f"分区数: {df.rdd.getNumPartitions()}")
print(f"总行数: {df.count()}")
print(f"总列数: {len(df.columns)}")
df.printSchema()

分区数: 2
总行数: 100000
总列数: 31
root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = tr

分区数: 2


总行数: 100000
总列数: 31
root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-

In [4]:
from pyspark.sql import functions as F

# 正负样本分布
df.groupBy("Class").count().show()

# 描述性统计
numeric_cols = [f"V{i}" for i in range(1, 29)] + ["Amount"]
df.select(numeric_cols).describe().show()
df.show(2)

+-----+-----+
|Class|count|
+-----+-----+
|    0|85536|
|    1|14464|
+-----+-----+



26/06/20 13:35:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|summary|                  V1|                 V2|                  V3|                  V4|                  V5|                  V6|                  V7|                  V8|                  V9|                 V10|                 V11|                 V12|                 V13|                 V14|                 V15|                 V16|                 V17|                 

+-------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|summary|                  V1|                 V2|                  V3|                  V4|                  V5|                  V6|                  V7|                  V8|                  V9|                 V10|                 V11|                 V12|                 V13|                 V14|                 V15|                 V16|                 V17|                 

## 2. 特征工程

In [5]:
from pyspark.ml.feature import VectorAssembler

# 特征列
feature_cols = [f"V{i}" for i in range(1, 29)] + ["Amount"]

# 检查 null
null_check = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in feature_cols])
print("Null counts per feature:")
null_check.show()

# 向量化（XGBoost 是树模型，不需要标准化）
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)

df_transformed = assembler.transform(df).select("features", "Class")
print("预处理完成！")
df_transformed.show(5, truncate=False)

Null counts per feature:


+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+
| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|
+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+
|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|
+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+

预处理完成！
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                                                                                                                                                                                                                                                                                                 

In [6]:
# 训练/测试集划分
train_df, test_df = df_transformed.randomSplit([0.8, 0.2], seed=42)

# 重新分区并缓存
train_df = train_df.repartition(2).cache()
test_df = test_df.cache()

# 触发缓存并统计
train_count = train_df.count()
test_count = test_df.count()

train_pos = train_df.filter(F.col("Class") == 1).count()
train_neg = train_df.filter(F.col("Class") == 0).count()
test_pos = test_df.filter(F.col("Class") == 1).count()
test_neg = test_df.filter(F.col("Class") == 0).count()

print(f"训练集: {train_count} (正:{train_pos}, 负:{train_neg})")
print(f"测试集: {test_count} (正:{test_pos}, 负:{test_neg})")

# 验证各分区不为空
partitions = train_df.rdd.glom().map(len).collect()
print(f"训练集各分区行数: {partitions}")

训练集: 80009 (正:11499, 负:68510)
测试集: 19991 (正:2965, 负:17026)


[Stage 45:>                                                         (0 + 2) / 2]

训练集各分区行数: [40004, 40005]


训练集各分区行数: [40004, 40005]


## 3. 分布式 XGBoost 训练

In [7]:
from xgboost.spark import SparkXGBClassifier
import time

# 配置分布式XGBoost分类器
xgb_classifier = SparkXGBClassifier(
    features_col="features",
    label_col="Class",
    num_workers=2,
    n_estimators=50,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="auc",
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=3.0,
    missing=0.0,
    verbosity=1,
)

print("开始分布式训练...")
start_time = time.time()

xgb_model = xgb_classifier.fit(train_df)

train_time = time.time() - start_time
print(f"训练完成！耗时: {train_time:.2f} 秒")

/usr/local/lib/python3.8/dist-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


开始分布式训练...


2026-06-20 13:35:21,232 INFO XGBoost-PySpark: _fit Running xgboost-2.1.4 on 2 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 0.8, 'device': 'cpu', 'eval_metric': 'auc', 'learning_rate': 0.1, 'max_depth': 6, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'scale_pos_weight': 3.0, 'subsample': 0.8, 'verbosity': 1, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}
2026-06-20 13:35:25,972 INFO XGBoost-PySpark: _fit Finished xgboost training!   


训练完成！耗时: 5.23 秒


2026-06-20 13:34:21,394 INFO XGBoost-PySpark: _fit Finished xgboost training!


训练完成！耗时: 4.93 秒


## 4. 模型评估

In [8]:
# 分布式预测
predictions = xgb_model.transform(test_df)
predictions.select("features", "Class", "rawPrediction", "probability", "prediction").show(10, truncate=False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+----------------------------------------+-----------------------------------------+----------+
|features                                                                                                                                                                                                                                                                                                                          

In [9]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

auc = BinaryClassificationEvaluator(
    labelCol="Class", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(predictions)

pr_auc = BinaryClassificationEvaluator(
    labelCol="Class", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
).evaluate(predictions)

accuracy = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)

f1 = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction", metricName="f1"
).evaluate(predictions)

print("=" * 50)
print("模型评估结果")
print("=" * 50)
print(f"AUC (ROC):      {auc:.4f}")
print(f"AUC (PR):       {pr_auc:.4f}")
print(f"Accuracy:       {accuracy:.4f}")
print(f"F1 Score:       {f1:.4f}")

模型评估结果
AUC (ROC):      0.9717
AUC (PR):       0.9466
Accuracy:       0.9679
F1 Score:       0.9685


模型评估结果
AUC (ROC):      0.9717
AUC (PR):       0.9465
Accuracy:       0.9671
F1 Score:       0.9677


In [10]:
# 分布式计算混淆矩阵
cm = predictions.groupBy("Class", "prediction").count().orderBy("Class", "prediction")
cm_pd = cm.toPandas()

tp = cm_pd[(cm_pd["Class"] == 1) & (cm_pd["prediction"] == 1)]["count"].sum()
tn = cm_pd[(cm_pd["Class"] == 0) & (cm_pd["prediction"] == 0)]["count"].sum()
fp = cm_pd[(cm_pd["Class"] == 0) & (cm_pd["prediction"] == 1)]["count"].sum()
fn = cm_pd[(cm_pd["Class"] == 1) & (cm_pd["prediction"] == 0)]["count"].sum()

print("混淆矩阵:")
print(f"                预测负      预测正")
print(f"实际负        {int(tn):>8}    {int(fp):>8}")
print(f"实际正        {int(fn):>8}    {int(tp):>8}")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

混淆矩阵:
                预测负      预测正
实际负           16576         450
实际正             191        2774

Precision: 0.8604
Recall:    0.9356


## 5. 模型预测（新数据）

In [11]:
import numpy as np
import pandas as pd

np.random.seed(123)
n_new = 5

new_X = np.random.randn(n_new, 28) * 0.5
new_amount = np.abs(np.random.randn(n_new) * 50 + 100)

new_data = np.column_stack([np.arange(n_new), new_X, new_amount, np.zeros(n_new)])
columns = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
new_pd = pd.DataFrame(new_data, columns=columns)
new_pd["Class"] = new_pd["Class"].astype(int)

new_spark_df = spark.createDataFrame(new_pd)
new_transformed = assembler.transform(new_spark_df).select("features")

new_predictions = xgb_model.transform(new_transformed)

print("新数据预测结果:")
result = new_predictions.select("probability", "prediction").toPandas()
for i, row in result.iterrows():
    fraud_prob = row["probability"][1]
    pred_label = "欺诈" if row["prediction"] == 1 else "正常"
    print(f"样本{i+1}: 欺诈概率={fraud_prob:.4f}, 预测={pred_label}")

新数据预测结果:
样本1: 欺诈概率=0.0319, 预测=正常
样本2: 欺诈概率=0.0196, 预测=正常
样本3: 欺诈概率=0.1245, 预测=正常
样本4: 欺诈概率=0.0199, 预测=正常
样本5: 欺诈概率=0.0307, 预测=正常


样本1: 欺诈概率=0.0311, 预测=正常
样本2: 欺诈概率=0.0210, 预测=正常
样本3: 欺诈概率=0.1211, 预测=正常
样本4: 欺诈概率=0.0234, 预测=正常
样本5: 欺诈概率=0.0284, 预测=正常


## 6. 总结

本Notebook演示了完整的 **PySpark + XGBoost 分布式机器学习流程**：

| 步骤 | 内容 | 分布式特性 |
|------|------|------------|
| 数据加载 | CSV → Spark DataFrame | 数据自动分区到多worker |
| 数据探索 | 描述性统计、类别分布 | 各worker并行计算聚合 |
| 特征工程 | VectorAssembler | 无需标准化（树模型） |
| 训练/测试划分 | randomSplit | 分布式shuffle |
| 模型训练 | SparkXGBClassifier | 多worker并行训练XGBoost |
| 模型评估 | AUC、PR-AUC、F1、混淆矩阵 | 分布式预测+聚合 |
| 新数据预测 | VectorAssembler + XGBoost | 分布式推理 |

In [12]:
# spark.stop()  # 取消注释以释放资源